# Customer Churn Prediction - Kaggle Challenge

This notebook provides a complete end-to-end solution for the Kaggle Churn Prediction Challenge.

## Table of Contents
1. Import Libraries
2. Load Data
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Feature Engineering
6. Model Training
7. Model Evaluation
8. Hyperparameter Tuning
9. Final Predictions & Submission

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully!")

## 2. Load Data

Load the training and test datasets from Kaggle. Make sure you've downloaded the data files.

In [ ]:
# Load training data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

# Display first few rows
train_df.head()

In [ ]:
# Check data info
print("\n=== Training Data Info ===")
train_df.info()

print("\n=== Test Data Info ===")
test_df.info()

In [ ]:
# Check for target variable distribution
if 'Churn' in train_df.columns:
    print("\nTarget Variable Distribution:")
    print(train_df['Churn'].value_counts())
    print("\nPercentage:")
    print(train_df['Churn'].value_counts(normalize=True) * 100)
    
    # Visualize
    plt.figure(figsize=(8, 5))
    train_df['Churn'].value_counts().plot(kind='bar')
    plt.title('Churn Distribution')
    plt.xlabel('Churn (0 = No, 1 = Yes)')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary
train_df.describe()

In [ ]:
# Check for missing values
print("Missing values in training data:")
missing_train = train_df.isnull().sum()
missing_train_pct = (missing_train / len(train_df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_train,
    'Percentage': missing_train_pct
})
print(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False))

print("\nMissing values in test data:")
missing_test = test_df.isnull().sum()
print(missing_test[missing_test > 0].sort_values(ascending=False))

In [ ]:
# Identify numerical and categorical columns
numerical_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()

# Remove target and ID columns if present
if 'Churn' in numerical_cols:
    numerical_cols.remove('Churn')
if 'CustomerID' in categorical_cols:
    categorical_cols.remove('CustomerID')
if 'Customer ID' in categorical_cols:
    categorical_cols.remove('Customer ID')

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols[:10]}...")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols[:10]}...")

In [ ]:
# Visualize numerical features distribution
if len(numerical_cols) > 0:
    fig, axes = plt.subplots(nrows=(len(numerical_cols[:12]) + 3) // 4, ncols=4, figsize=(20, 15))
    axes = axes.flatten()
    
    for idx, col in enumerate(numerical_cols[:12]):
        train_df[col].hist(bins=30, ax=axes[idx], edgecolor='black')
        axes[idx].set_title(f'Distribution of {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
    
    # Hide empty subplots
    for idx in range(len(numerical_cols[:12]), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix for numerical features
if len(numerical_cols) > 0:
    # Select subset for better visualization
    corr_cols = numerical_cols[:15] if len(numerical_cols) > 15 else numerical_cols
    
    plt.figure(figsize=(14, 10))
    correlation_matrix = train_df[corr_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                square=True, linewidths=0.5)
    plt.title('Correlation Matrix of Numerical Features')
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze categorical variables vs Churn
if 'Churn' in train_df.columns and len(categorical_cols) > 0:
    fig, axes = plt.subplots(nrows=(len(categorical_cols[:8]) + 1) // 2, ncols=2, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx, col in enumerate(categorical_cols[:8]):
        if train_df[col].nunique() < 20:  # Only plot if not too many unique values
            churn_by_cat = pd.crosstab(train_df[col], train_df['Churn'], normalize='index') * 100
            churn_by_cat.plot(kind='bar', ax=axes[idx], stacked=False)
            axes[idx].set_title(f'Churn Rate by {col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Percentage (%)')
            axes[idx].legend(['No Churn', 'Churn'])
            axes[idx].tick_params(axis='x', rotation=45)
    
    # Hide empty subplots
    for idx in range(len(categorical_cols[:8]), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 4. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
train_processed = train_df.copy()
test_processed = test_df.copy()

# Store IDs for submission
test_ids = None
if 'CustomerID' in test_processed.columns:
    test_ids = test_processed['CustomerID']
elif 'Customer ID' in test_processed.columns:
    test_ids = test_processed['Customer ID']

print("Data copied for preprocessing")

In [ ]:
# Handle missing values

# For numerical columns: fill with median
for col in numerical_cols:
    if train_processed[col].isnull().sum() > 0:
        median_value = train_processed[col].median()
        train_processed[col].fillna(median_value, inplace=True)
        if col in test_processed.columns:
            test_processed[col].fillna(median_value, inplace=True)

# For categorical columns: fill with mode or 'Unknown'
for col in categorical_cols:
    if train_processed[col].isnull().sum() > 0:
        mode_value = train_processed[col].mode()[0] if not train_processed[col].mode().empty else 'Unknown'
        train_processed[col].fillna(mode_value, inplace=True)
        if col in test_processed.columns:
            test_processed[col].fillna(mode_value, inplace=True)

print("Missing values handled")
print(f"Remaining missing values in train: {train_processed.isnull().sum().sum()}")
print(f"Remaining missing values in test: {test_processed.isnull().sum().sum()}")

In [ ]:
# Remove ID columns
id_columns = ['CustomerID', 'Customer ID', 'ID', 'id']
for col in id_columns:
    if col in train_processed.columns:
        train_processed.drop(col, axis=1, inplace=True)
    if col in test_processed.columns:
        test_processed.drop(col, axis=1, inplace=True)

print("ID columns removed")

## 5. Feature Engineering

In [ ]:
# Encode categorical variables

# Use Label Encoding for binary categorical variables
# Use One-Hot Encoding for multi-class categorical variables

label_encoders = {}
categorical_cols_updated = train_processed.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols_updated:
    unique_values = train_processed[col].nunique()
    
    if unique_values == 2:
        # Binary encoding
        le = LabelEncoder()
        train_processed[col] = le.fit_transform(train_processed[col].astype(str))
        if col in test_processed.columns:
            # Handle unseen labels
            test_processed[col] = test_processed[col].astype(str).map(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )
        label_encoders[col] = le
    elif unique_values <= 10:
        # One-hot encoding for categorical with few categories
        train_dummies = pd.get_dummies(train_processed[col], prefix=col, drop_first=True)
        train_processed = pd.concat([train_processed, train_dummies], axis=1)
        train_processed.drop(col, axis=1, inplace=True)
        
        if col in test_processed.columns:
            test_dummies = pd.get_dummies(test_processed[col], prefix=col, drop_first=True)
            test_processed = pd.concat([test_processed, test_dummies], axis=1)
            test_processed.drop(col, axis=1, inplace=True)
            
            # Align columns
            missing_cols = set(train_dummies.columns) - set(test_dummies.columns)
            for missing_col in missing_cols:
                test_processed[missing_col] = 0
    else:
        # Label encoding for high cardinality
        le = LabelEncoder()
        train_processed[col] = le.fit_transform(train_processed[col].astype(str))
        if col in test_processed.columns:
            test_processed[col] = test_processed[col].astype(str).map(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )
        label_encoders[col] = le

print("Categorical encoding completed")
print(f"Training data shape after encoding: {train_processed.shape}")
print(f"Test data shape after encoding: {test_processed.shape}")

In [ ]:
# Separate features and target
if 'Churn' in train_processed.columns:
    X = train_processed.drop('Churn', axis=1)
    y = train_processed['Churn']
else:
    # If target column name is different, adjust accordingly
    target_col = [col for col in train_processed.columns if 'churn' in col.lower()]
    if target_col:
        X = train_processed.drop(target_col[0], axis=1)
        y = train_processed[target_col[0]]
    else:
        print("Warning: Target column not found!")
        X = train_processed
        y = None

X_test = test_processed.copy()

# Ensure test has same columns as train
missing_cols = set(X.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0

extra_cols = set(X_test.columns) - set(X.columns)
X_test = X_test.drop(columns=list(extra_cols))

# Reorder columns to match
X_test = X_test[X.columns]

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape if y is not None else 'N/A'}")
print(f"Test features shape: {X_test.shape}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Feature scaling completed")

In [ ]:
# Train-validation split
if y is not None:
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    
    print(f"Training set size: {X_train.shape}")
    print(f"Validation set size: {X_val.shape}")
    print(f"\nClass distribution in training set:")
    print(y_train.value_counts(normalize=True))

## 6. Model Training

In [ ]:
# Define models to train
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=5),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=5, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=5, verbose=-1)
}

print("Models defined successfully")

In [ ]:
# Train and evaluate models
results = {}

if y is not None:
    for name, model in models.items():
        print(f"\n{'='*60}")
        print(f"Training {name}...")
        print('='*60)
        
        # Train model
        model.fit(X_train, y_train)
        
        # Predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        
        # Metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        val_acc = accuracy_score(y_val, y_val_pred)
        val_precision = precision_score(y_val, y_val_pred, average='macro')
        val_recall = recall_score(y_val, y_val_pred, average='macro')
        val_f1 = f1_score(y_val, y_val_pred, average='macro')
        
        # Store results
        results[name] = {
            'model': model,
            'train_accuracy': train_acc,
            'val_accuracy': val_acc,
            'precision': val_precision,
            'recall': val_recall,
            'f1_score': val_f1
        }
        
        print(f"Training Accuracy: {train_acc:.4f}")
        print(f"Validation Accuracy: {val_acc:.4f}")
        print(f"Precision: {val_precision:.4f}")
        print(f"Recall: {val_recall:.4f}")
        print(f"F1-Score: {val_f1:.4f}")

## 7. Model Evaluation

In [ ]:
# Compare model performance
if results:
    results_df = pd.DataFrame({
        'Model': list(results.keys()),
        'Train Accuracy': [results[m]['train_accuracy'] for m in results],
        'Val Accuracy': [results[m]['val_accuracy'] for m in results],
        'Precision': [results[m]['precision'] for m in results],
        'Recall': [results[m]['recall'] for m in results],
        'F1-Score': [results[m]['f1_score'] for m in results]
    })
    
    results_df = results_df.sort_values('F1-Score', ascending=False)
    print("\n" + "="*80)
    print("MODEL COMPARISON")
    print("="*80)
    print(results_df.to_string(index=False))
    
    # Visualize results
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Accuracy comparison
    results_df.plot(x='Model', y=['Train Accuracy', 'Val Accuracy'], 
                    kind='bar', ax=axes[0], rot=45)
    axes[0].set_title('Model Accuracy Comparison')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend(['Train', 'Validation'])
    axes[0].grid(axis='y', alpha=0.3)
    
    # Metrics comparison
    results_df.plot(x='Model', y=['Precision', 'Recall', 'F1-Score'], 
                    kind='bar', ax=axes[1], rot=45)
    axes[1].set_title('Model Metrics Comparison')
    axes[1].set_ylabel('Score')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Detailed evaluation of best model
if results:
    best_model_name = results_df.iloc[0]['Model']
    best_model = results[best_model_name]['model']
    
    print(f"\nBest Model: {best_model_name}")
    print("="*60)
    
    # Confusion Matrix
    y_val_pred = best_model.predict(X_val)
    cm = confusion_matrix(y_val, y_val_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    plt.title(f'Confusion Matrix - {best_model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_val, y_val_pred, 
                               target_names=['No Churn', 'Churn']))

In [ ]:
# Feature importance (for tree-based models)
if results and hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(20)
    
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'])
    plt.xlabel('Importance')
    plt.title(f'Top 20 Feature Importances - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\nTop 20 Important Features:")
    print(feature_importance.to_string(index=False))

## 8. Hyperparameter Tuning (Optional)

Fine-tune the best performing model

In [ ]:
# Example: Tune XGBoost parameters
from sklearn.model_selection import GridSearchCV

if y is not None:
    print("Starting hyperparameter tuning for XGBoost...")
    print("This may take several minutes...\n")
    
    # Define parameter grid
    param_grid = {
        'max_depth': [3, 5, 7],
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'subsample': [0.8, 1.0]
    }
    
    # Grid search
    xgb_model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')
    grid_search = GridSearchCV(
        xgb_model, 
        param_grid, 
        cv=3, 
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    # Evaluate on validation set
    tuned_model = grid_search.best_estimator_
    y_val_pred_tuned = tuned_model.predict(X_val)
    
    print("\nTuned Model Performance:")
    print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred_tuned):.4f}")
    print(f"F1-Score: {f1_score(y_val, y_val_pred_tuned, average='macro'):.4f}")
    print(f"Recall: {recall_score(y_val, y_val_pred_tuned, average='macro'):.4f}")

## 9. Final Predictions & Submission

In [ ]:
# Train final model on full dataset
if y is not None:
    print("Training final model on complete training data...")
    
    # Use the best model (or tuned model if available)
    try:
        final_model = tuned_model
        print("Using tuned XGBoost model")
    except:
        final_model = best_model
        print(f"Using {best_model_name}")
    
    # Retrain on full data
    final_model.fit(X_scaled, y)
    
    print("Final model training completed!")

In [ ]:
# Make predictions on test set
if y is not None:
    print("Making predictions on test set...")
    
    test_predictions = final_model.predict(X_test_scaled)
    test_predictions_proba = final_model.predict_proba(X_test_scaled)
    
    print(f"\nPredictions completed!")
    print(f"Total predictions: {len(test_predictions)}")
    print(f"Predicted churn count: {test_predictions.sum()}")
    print(f"Predicted churn rate: {test_predictions.mean()*100:.2f}%")

In [ ]:
# Create submission file
if y is not None and test_ids is not None:
    submission = pd.DataFrame({
        'CustomerID': test_ids,  # Adjust column name as needed
        'Churn': test_predictions
    })
    
    # Save submission
    submission.to_csv('submission.csv', index=False)
    
    print("\nSubmission file created: submission.csv")
    print("\nFirst few rows of submission:")
    print(submission.head(10))
    print(f"\nSubmission shape: {submission.shape}")
else:
    print("Warning: Could not create submission file. Check if test IDs are available.")

In [ ]:
# Optional: Create ensemble predictions
if y is not None and len(results) > 1:
    print("\nCreating ensemble predictions...")
    
    # Get predictions from top 3 models
    top_models = results_df.head(3)['Model'].tolist()
    ensemble_predictions = []
    
    for model_name in top_models:
        model = results[model_name]['model']
        model.fit(X_scaled, y)  # Retrain on full data
        pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        ensemble_predictions.append(pred_proba)
    
    # Average predictions
    ensemble_proba = np.mean(ensemble_predictions, axis=0)
    ensemble_pred = (ensemble_proba > 0.5).astype(int)
    
    # Create ensemble submission
    if test_ids is not None:
        ensemble_submission = pd.DataFrame({
            'CustomerID': test_ids,
            'Churn': ensemble_pred
        })
        
        ensemble_submission.to_csv('submission_ensemble.csv', index=False)
        print("Ensemble submission file created: submission_ensemble.csv")
        print(f"Ensemble predicted churn rate: {ensemble_pred.mean()*100:.2f}%")

## Summary

This notebook provides a complete pipeline for customer churn prediction:

1. **Data Loading**: Loaded training and test datasets
2. **EDA**: Explored data distribution, correlations, and patterns
3. **Preprocessing**: Handled missing values, encoded categorical variables
4. **Feature Engineering**: Scaled features and created train/validation splits
5. **Model Training**: Trained multiple models (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM)
6. **Evaluation**: Compared model performance using various metrics
7. **Hyperparameter Tuning**: Optimized the best model
8. **Predictions**: Generated final predictions and submission file

**Next Steps:**
- Upload `submission.csv` to Kaggle
- Try additional feature engineering
- Experiment with different model ensembles
- Handle class imbalance if present (SMOTE, class weights)
- Cross-validation for more robust evaluation